# 06. Exercise Definition Test

This notebook validates the exercise definition loader against all 5 YAML files
in `data/exercise_definitions/`.

Tests covered:

1. Load all definitions — 5 YAMLs load without errors
2. Required field validation — missing field raises `ValueError`
3. Vocabulary validation — out-of-vocabulary value emits a warning
4. Phase ratio validation — ratio sum ≠ 1.0 emits a warning
5. Generic fallback — None exercise_id loads generic
6. Missing file fallback — unknown exercise_id falls back to generic
7. Pipeline step integration — `exercise_definition` step appears in report

This notebook assumes that the previous checks are already working:

- 00_environment_check through 05_annotation_mask_test

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings
from pathlib import Path

from movement.exercise_definition import (
    ExerciseDefinition,
    load_all_exercise_definitions,
    load_exercise_definition,
)

DEFINITIONS_DIR = Path("../data/exercise_definitions")

## Case 1: Load All Definitions

All 5 YAML files (`squat`, `lunge`, `pike_pushup`, `plank_shoulder_tap`, `generic`)
should load without errors or warnings.

In [3]:
all_defs = load_all_exercise_definitions(DEFINITIONS_DIR)

print(f"loaded {len(all_defs)} definitions:")
for ex_id, ed in all_defs.items():
    print(f"  {ex_id:25s}  v{ed.version}  fallback={ed.is_generic_fallback}")

loaded 5 definitions:
  generic                    v0.1.0  fallback=True
  lunge                      v0.1.0  fallback=False
  pike_pushup                v0.1.0  fallback=False
  plank_shoulder_tap         v0.1.0  fallback=False
  squat                      v0.1.0  fallback=False


In [4]:
expected_ids = {"squat", "lunge", "pike_pushup", "plank_shoulder_tap", "generic"}
assert set(all_defs.keys()) == expected_ids, f"unexpected keys: {set(all_defs.keys())}"
assert all_defs["generic"].is_generic_fallback is True
assert all(not ed.is_generic_fallback for k, ed in all_defs.items() if k != "generic")
print("PASS: all 5 definitions loaded correctly")

PASS: all 5 definitions loaded correctly


## Case 2: Per-Definition Field Inspection

Spot-check the most important typed fields for each definition.

In [5]:
for ex_id, ed in all_defs.items():
    clf = ed.classification
    print(f"── {ex_id} ──────────────────────────────")
    print(f"  laterality       : {clf.get('laterality')}")
    print(f"  posture_type     : {clf.get('posture_type')}")
    print(f"  primary_plane    : {clf.get('primary_plane')}")
    print(f"  phase_model.type : {ed.phase_model.type}")
    print(f"  expected_ratio   : {ed.phase_model.expected_ratio}")
    print(f"  primary_joints   : {ed.landmarks.primary_joints}")
    print(f"  compensation_candidates ({len(ed.compensation_candidates)}): {ed.compensation_candidates}")
    print()

── generic ──────────────────────────────
  laterality       : bilateral_symmetric
  posture_type     : standing
  primary_plane    : sagittal
  phase_model.type : cyclic
  expected_ratio   : {}
  primary_joints   : []
  compensation_candidates (0): []

── lunge ──────────────────────────────
  laterality       : alternating
  posture_type     : standing_split
  primary_plane    : sagittal
  phase_model.type : resistance_phase
  expected_ratio   : {'eccentric': 0.45, 'isometric': 0.05, 'concentric': 0.5}
  primary_joints   : ['left_hip', 'right_hip', 'left_knee', 'right_knee', 'left_ankle', 'right_ankle']
  compensation_candidates (12): ['knee_valgus', 'asymmetric_knee_flexion', 'asymmetric_hip_flexion', 'insufficient_rear_hip_extension', 'excessive_trunk_flexion', 'lateral_trunk_lean', 'lateral_pelvic_shift', 'pelvis_drop', 'unstable_step_width', 'heel_lift', 'tempo_instability', 'phase_timing_asymmetry']

── pike_pushup ──────────────────────────────
  laterality       : bilateral_sy

## Case 3: Required Field Validation

A YAML missing a required field must raise `ValueError`.

In [6]:
import tempfile
import yaml

# Write a minimal YAML missing the 'landmarks' field
bad_yaml = {
    "exercise_id": "test_incomplete",
    "classification": {"family": "lower_body", "laterality": "bilateral_symmetric",
                       "posture_type": "standing", "kinetic_chain": "closed_chain",
                       "primary_plane": "sagittal"},
    "phase_model": {"type": "resistance_phase", "expected_ratio": {"eccentric": 0.5, "concentric": 0.5}},
    # 'landmarks' intentionally omitted
    "compensation_candidates": [],
    "feature_domains": {"spatial": [], "temporal": [], "control": [], "biomechanical_proxy": []},
    "quality_rules": {"minimum_visible_landmark_ratio": 0.8},
}

with tempfile.TemporaryDirectory() as tmpdir:
    p = Path(tmpdir) / "test_incomplete.yaml"
    p.write_text(yaml.dump(bad_yaml), encoding="utf-8")
    # also copy generic fallback so the dir is valid
    import shutil
    shutil.copy(DEFINITIONS_DIR / "generic.yaml", Path(tmpdir) / "generic.yaml")

    try:
        load_exercise_definition("test_incomplete", tmpdir)
        print("FAIL: expected ValueError was not raised")
    except ValueError as e:
        print("PASS: ValueError raised as expected")
        print(f"  {e}")

PASS: ValueError raised as expected
  Exercise definition 'C:\Users\andi9\AppData\Local\Temp\tmpzyn1hxl2\test_incomplete.yaml' failed validation:
  - missing required field: 'landmarks'


## Case 4: Vocabulary Warning

An out-of-vocabulary value must emit a `UserWarning` but still load successfully.

In [7]:
bad_vocab_yaml = {
    "exercise_id": "test_vocab",
    "classification": {
        "family": "lower_body",
        "laterality": "bilateral_symmetric",
        "posture_type": "INVALID_POSTURE",  # out-of-vocabulary
        "kinetic_chain": "closed_chain",
        "primary_plane": "sagittal",
    },
    "phase_model": {"type": "resistance_phase",
                   "expected_ratio": {"eccentric": 0.5, "concentric": 0.5}},
    "landmarks": {"model": "mediapipe_pose_33", "primary_joints": ["left_hip"],
                  "critical_landmarks": [23]},
    "compensation_candidates": [],
    "feature_domains": {"spatial": [], "temporal": [], "control": [], "biomechanical_proxy": []},
    "quality_rules": {"minimum_visible_landmark_ratio": 0.8},
}

with tempfile.TemporaryDirectory() as tmpdir:
    p = Path(tmpdir) / "test_vocab.yaml"
    p.write_text(yaml.dump(bad_vocab_yaml), encoding="utf-8")
    shutil.copy(DEFINITIONS_DIR / "generic.yaml", Path(tmpdir) / "generic.yaml")

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        ed = load_exercise_definition("test_vocab", tmpdir)

    vocab_warns = [w for w in caught if "not in controlled vocabulary" in str(w.message)]
    print(f"PASS: loaded successfully with {len(vocab_warns)} vocabulary warning(s)")
    for w in vocab_warns:
        print(f"  {w.message}")

PASS: loaded successfully with 1 vocabulary warning(s)
  [test_vocab] 'classification.posture_type' value 'INVALID_POSTURE' is not in controlled vocabulary


## Case 5: Phase Ratio Warning

When `resistance_phase` or `task_phase` ratios do not sum to ≈ 1.0,
a `UserWarning` must be emitted.

In [8]:
bad_ratio_yaml = {
    "exercise_id": "test_ratio",
    "classification": {"family": "lower_body", "laterality": "bilateral_symmetric",
                       "posture_type": "standing", "kinetic_chain": "closed_chain",
                       "primary_plane": "sagittal"},
    "phase_model": {
        "type": "resistance_phase",
        "expected_ratio": {"eccentric": 0.3, "isometric": 0.1, "concentric": 0.3},  # sum = 0.7
    },
    "landmarks": {"model": "mediapipe_pose_33", "primary_joints": ["left_hip"],
                  "critical_landmarks": [23]},
    "compensation_candidates": [],
    "feature_domains": {"spatial": [], "temporal": [], "control": [], "biomechanical_proxy": []},
    "quality_rules": {"minimum_visible_landmark_ratio": 0.8},
}

with tempfile.TemporaryDirectory() as tmpdir:
    p = Path(tmpdir) / "test_ratio.yaml"
    p.write_text(yaml.dump(bad_ratio_yaml), encoding="utf-8")
    shutil.copy(DEFINITIONS_DIR / "generic.yaml", Path(tmpdir) / "generic.yaml")

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        ed = load_exercise_definition("test_ratio", tmpdir)

    ratio_warns = [w for w in caught if "expected_ratio sums to" in str(w.message)]
    print(f"PASS: {len(ratio_warns)} phase ratio warning(s) emitted")
    for w in ratio_warns:
        print(f"  {w.message}")

PASS: 1 phase ratio warning(s) emitted
  [test_ratio] phase_model.expected_ratio sums to 0.700, expected 1.0 ± 0.02


## Case 6: Generic Fallback — None exercise_id

Passing `None` as `exercise_id` must load the generic definition and set
`is_generic_fallback=True`.

In [9]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    generic_def = load_exercise_definition(None, DEFINITIONS_DIR)

fallback_warns = [w for w in caught if "generic fallback" in str(w.message).lower()]

assert generic_def.exercise_id == "generic"
assert generic_def.is_generic_fallback is True
assert len(fallback_warns) >= 1
print("PASS: None exercise_id → generic fallback with warning")
print(f"  warning: {fallback_warns[0].message}")

PASS: None exercise_id → generic fallback with warning


## Case 7: Generic Fallback — Missing File

An unknown `exercise_id` whose YAML does not exist must silently fall back
to generic and set `is_generic_fallback=True`.

In [10]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    fallback_def = load_exercise_definition("nonexistent_exercise", DEFINITIONS_DIR)

fallback_warns = [w for w in caught if "not found" in str(w.message).lower()]

assert fallback_def.exercise_id == "generic"
assert fallback_def.is_generic_fallback is True
assert len(fallback_warns) >= 1
print("PASS: missing YAML → generic fallback with warning")
print(f"  warning: {fallback_warns[0].message}")

PASS: missing YAML → generic fallback with warning


## Case 8: Pipeline Step Integration

Running the pipeline with `exercise_definition.enabled: true` must add
an `'exercise_definition'` key to the report dict.

In [11]:
from pathlib import Path

from movement.annotation import load_annotation_csv
from movement.config import LANDMARKS
from movement.io import load_pose_csv
from movement.pipeline import load_pipeline_config, run_pipeline

config_path = Path("../configs/pipeline_default.yaml")
csv_path = "../data/sample/mediapipe_squat_synthetic.csv"
ann_path = "../data/sample/mediapipe_squat_synthetic_annotation.csv"

config = load_pipeline_config(config_path)
df = load_pose_csv(csv_path)
ann_df = load_annotation_csv(ann_path)

print("exercise_definition.enabled:", config.exercise_definition.enabled)
print("exercise_definition.definitions_dir:", config.exercise_definition.definitions_dir)
print("exercise_definition.exercise_id:", config.exercise_definition.exercise_id)

exercise_definition.enabled: True
exercise_definition.definitions_dir: data/exercise_definitions
exercise_definition.exercise_id: None


In [12]:
import warnings as _w

with _w.catch_warnings(record=True) as caught:
    _w.simplefilter("always")
    result_df, report = run_pipeline(df, config=config, landmarks=LANDMARKS, ann_df=ann_df)

print("steps executed:", list(report.keys()))
print()

if caught:
    print(f"{len(caught)} warning(s) during pipeline run:")
    for w in caught:
        print(f"  [{w.category.__name__}] {w.message}")

steps executed: ['validation', 'exercise_definition', 'normalization']

1 warning(s) during pipeline run:
  [UserWarning] exercise_id is None or empty — loading generic fallback definition. Biomarker output will be restricted to exercise-agnostic features.


In [13]:
import json

assert "exercise_definition" in report, "exercise_definition step missing from report"
exd_report = report["exercise_definition"]
print(json.dumps(exd_report, indent=2))

# The sample data has exercise_type='squat' (one of the 4 target exercises)
# so the squat YAML is loaded directly and is_generic_fallback should be False
print()
print(f"PASS: exercise_definition step in report")
print(f"  exercise_id        : {exd_report['exercise_id']}")
print(f"  is_generic_fallback: {exd_report['is_generic_fallback']}")

{
  "exercise_id": "generic",
  "display_name": "Generic Movement",
  "version": "0.1.0",
  "is_generic_fallback": true,
  "laterality": "bilateral_symmetric",
  "primary_plane": "sagittal",
  "compensation_candidates": []
}

PASS: exercise_definition step in report
  exercise_id        : generic
  is_generic_fallback: True


## Interpretation

Expected results:

**Case 1 — Load all:**
- 5 definitions loaded: squat, lunge, pike_pushup, plank_shoulder_tap, generic
- Only generic has `is_generic_fallback=True`

**Case 2 — Field inspection:**
- `laterality` values span bilateral_symmetric, alternating (lunge, plank_shoulder_tap)
- `posture_type` values span standing, inverted_closed_chain, plank
- generic has empty `primary_joints` and `compensation_candidates`

**Case 3 — Required field error:**
- `ValueError` raised when `landmarks` field is absent

**Case 4 — Vocabulary warning:**
- `UserWarning` emitted for invalid posture_type; definition still loads

**Case 5 — Phase ratio warning:**
- `UserWarning` emitted when ratio sum ≠ 1.0 ± 0.02; definition still loads

**Case 6 — None fallback:**
- `exercise_id=None` → generic loaded, warning emitted

**Case 7 — Missing file fallback:**
- Unknown `exercise_id` → generic loaded, warning emitted

**Case 8 — Pipeline integration:**
- `exercise_definition` key appears in pipeline report
- Sample data uses `exercise_type='squat'` so the squat YAML is loaded directly,
  and `is_generic_fallback` is `False`
- The generic-fallback path is still exercised by Cases 6 and 7 above